# Opening-range gap study

Reads the completed OR-line retest trade ledger from Drive. This does not change entries or replay trades. Compare QQQ opening gap magnitude >1% versus <=1%, and each stock's opening gap <=1%, >1–2%, and >2%. Absolute gap magnitude is used; up/down direction is retained separately. The gap is 09:30 ET open versus the preceding **regular-session** 15:59 ET close, known before any signal.

A ticker prior close is accepted only if it is on the prior QQQ trading session. Missing first-session/prior-close data remain **Unknown**. The study uses the same 57 August-history tickers, excludes Nov 4–5, and preserves four strategy combinations (1/2-minute close × 1%/2% target) and both sides. Wins require full target, loss includes stop or timed loss <= -0.5%, otherwise neutral. Net R uses the original 1% stop and 5 bps slippage per side.

Reports W/L/N and net R by QQQ gap, ticker gap, joint gap, up/down direction, price group, earlier/later dates, and date. Gap date counts and missingness are included. Treat results as exploratory: a few QQQ >1% dates can generate many correlated stock trades.


In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display
import pandas as pd, numpy as np, re
drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/backtesting'),Path('/content/drive/MyDrive/Backtesting')] if p.is_dir()),None)
SOURCE=Path('/content/drive/MyDrive/Opening_Range_Line_Retest_1m_vs_2m')
OUT=Path('/content/drive/MyDrive/Opening_Range_Gap_Study')
OUT.mkdir(exist_ok=True)
if ROOT is None: raise FileNotFoundError('Set ROOT to Backtesting folder')
LEDGER=SOURCE/'filled_trade_ledger.csv'
if not LEDGER.exists(): raise FileNotFoundError(LEDGER)
trades=pd.read_csv(LEDGER,dtype={'ticker':str,'date':str})
trades=trades[~trades.date.isin(['2026-11-04','2026-11-05'])].copy()
COHORT=set(trades.ticker)
assert len(COHORT)==57, f'Expected 57 tickers, got {len(COHORT)}; inspect source ledger'
print('Trade rows:',len(trades),'| tickers:',len(COHORT))


In [ ]:
# Extract only the 09:30 open and 15:59 close from historical 1-minute files.
# Filename date is the pull date, not necessarily the trading session.
points={}
for i,p in enumerate(ROOT.rglob('*_1m_*.csv'),1):
    m=re.match(r'(.+)_1m_\d{4}-\d{2}-\d{2}',p.stem)
    if not m: continue
    ticker=m.group(1).upper()
    if ticker not in COHORT and ticker!='QQQ': continue
    try:
        a=pd.read_csv(p)
        col=next((c for c in a if c.lower() in ('datetime','date','timestamp')),a.columns[0])
        ts=pd.to_datetime(a[col],utc=True,errors='coerce').dt.tz_convert('America/New_York')
        hhmm=ts.dt.strftime('%H:%M')
        for clock,field in [('09:30','Open'),('15:59','Close')]:
            subset=a.loc[hhmm.eq(clock),[field]]
            for idx,row in subset.iterrows():
                day=str(ts.loc[idx].date())
                value=pd.to_numeric(row[field],errors='coerce')
                if pd.notna(value): points[(ticker,day,clock)]=float(value)
    except Exception as e: print('Skipped',p.name,str(e)[:100])
    if i%1000==0: print('Scanned files:',i)
qqq_dates=sorted({day for (ticker,day,clock) in points if ticker=='QQQ' and clock=='09:30'})
previous={day:qqq_dates[i-1] for i,day in enumerate(qqq_dates) if i>0}
print('QQQ sessions with 09:30:',len(qqq_dates),'| first/last:',qqq_dates[:1],qqq_dates[-1:])
needed=sorted(set(trades.date))
gap_rows=[]
for date in needed:
    prior=previous.get(date)
    qo=points.get(('QQQ',date,'09:30'))
    qc=points.get(('QQQ',prior,'15:59')) if prior else None
    qgap=100*(qo/qc-1) if qo and qc else np.nan
    for ticker in COHORT:
        op=points.get((ticker,date,'09:30'))
        close=points.get((ticker,prior,'15:59')) if prior else None
        tgap=100*(op/close-1) if op and close else np.nan
        gap_rows.append(dict(ticker=ticker,date=date,prior_session=prior,
                             qqq_gap_pct=qgap,ticker_gap_pct=tgap,
                             qqq_prior_close=qc,ticker_prior_close=close))
gap=pd.DataFrame(gap_rows)
gap.to_csv(OUT/'gap_coverage_ticker_date.csv',index=False)
trades=trades.merge(gap,on=['ticker','date'],how='left',validate='many_to_one')
print('Known QQQ gaps:',trades.qqq_gap_pct.notna().sum(),'/',len(trades))
print('Known ticker gaps:',trades.ticker_gap_pct.notna().sum(),'/',len(trades))


In [ ]:
trades['qqq_gap_band']=np.select(
    [trades.qqq_gap_pct.abs().gt(1),trades.qqq_gap_pct.notna()],
    ['>1%','<=1%'],default='Unknown')
trades['ticker_gap_band']=np.select(
    [trades.ticker_gap_pct.abs().gt(2),trades.ticker_gap_pct.abs().gt(1),trades.ticker_gap_pct.notna()],
    ['>2%','>1–2%','<=1%'],default='Unknown')
trades['qqq_gap_direction']=np.select(
    [trades.qqq_gap_pct.gt(0),trades.qqq_gap_pct.lt(0),trades.qqq_gap_pct.eq(0)],
    ['up','down','flat'],default='Unknown')
trades['ticker_gap_direction']=np.select(
    [trades.ticker_gap_pct.gt(0),trades.ticker_gap_pct.lt(0),trades.ticker_gap_pct.eq(0)],
    ['up','down','flat'],default='Unknown')
trades['period']=np.where(trades.date.lt('2026-10-07'),'earlier','later')
trades['weekday']=pd.to_datetime(trades.date).dt.day_name()
trades.to_csv(OUT/'trades_with_opening_gaps.csv',index=False)
def summary(by):
    x=trades.assign(win=trades.result.eq('win').astype(int),
                    loss=trades.result.eq('loss').astype(int),
                    neutral=trades.result.eq('neutral').astype(int))
    y=x.groupby(by,dropna=False).agg(trades=('R','size'),wins=('win','sum'),losses=('loss','sum'),
        neutrals=('neutral','sum'),net_R=('R','sum'),average_R=('R','mean'),
        tickers=('ticker','nunique'),dates=('date','nunique')).reset_index()
    assert (y.wins+y.losses+y.neutrals==y.trades).all()
    return y.round({'net_R':3,'average_R':4})
for name,by in [
    ('qqq_gap',['signal_minutes','plan','side','qqq_gap_band']),
    ('ticker_gap',['signal_minutes','plan','side','ticker_gap_band']),
    ('joint_gap',['signal_minutes','plan','side','qqq_gap_band','ticker_gap_band']),
    ('qqq_gap_direction',['signal_minutes','plan','side','qqq_gap_band','qqq_gap_direction']),
    ('ticker_gap_direction',['signal_minutes','plan','side','ticker_gap_band','ticker_gap_direction']),
    ('gap_period',['period','signal_minutes','plan','side','qqq_gap_band','ticker_gap_band']),
    ('gap_price_group',['signal_minutes','plan','side','price_group','ticker_gap_band']),
    ('qqq_gap_by_date',['date','qqq_gap_band','signal_minutes','plan','side'])]:
    table=summary(by)
    table.to_csv(OUT/(name+'.csv'),index=False)
    print('\n'+name);display(table.head(80))
print('Output:',OUT)
